<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day12_mozambique_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
 #Setup
!pip install earthengine-api geemap geopandas folium pandas numpy -q

import ee
import geemap
import pandas as pd
import numpy as np
import folium
import geopandas as gpd
from shapely.geometry import Point

ee.Authenticate()
ee.Initialize(project='tamil-nadu-flood-risk')

print(ee.String('Day 12 — Mozambique ready!').getInfo())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.2 MB/s eta 0:00:00
Day 12 — Mozambique ready!


In [5]:
# Mozambique school data
import pandas as pd
import numpy as np

# Sample Mozambique schools
# Key cities: Maputo, Beira, Nampula, Quelimane, Tete
mozambique_schools = {
    'school_name': [
        'Primary School Maputo Central',
        'Secondary School Beira Coast',
        'Primary School Quelimane',
        'Secondary School Nampula',
        'Primary School Tete',
        'School Inhambane Coast',
        'Primary School Sofala',
        'Secondary School Zambezia',
        'School Cabo Delgado',
        'Primary School Gaza'
    ],
    'latitude': [
        -25.96, -19.84, -17.88, -15.12, -16.16,
        -23.86, -19.52, -17.05, -12.37, -24.05
    ],
    'longitude': [
        32.57, 34.84, 36.89, 39.27, 33.59,
        35.38, 34.56, 36.98, 40.52, 34.40
    ],
    'connectivity': [
        'connected', 'none', 'none', 'connected',
        'none', 'none', 'none', 'none',
        'none', 'connected'
    ],
    'province': [
        'Maputo', 'Sofala', 'Zambezia', 'Nampula',
        'Tete', 'Inhambane', 'Sofala', 'Zambezia',
        'Cabo Delgado', 'Gaza'
    ]
}

moz_df = pd.DataFrame(mozambique_schools)
print(f'✅ {len(moz_df)} Mozambique schools loaded!')
print(moz_df[['school_name', 'province', 'connectivity']].to_string(index=False))

✅ 10 Mozambique schools loaded!
                  school_name     province connectivity
Primary School Maputo Central       Maputo    connected
 Secondary School Beira Coast       Sofala         none
     Primary School Quelimane     Zambezia         none
     Secondary School Nampula      Nampula    connected
          Primary School Tete         Tete         none
       School Inhambane Coast    Inhambane         none
        Primary School Sofala       Sofala         none
    Secondary School Zambezia     Zambezia         none
          School Cabo Delgado Cabo Delgado         none
          Primary School Gaza         Gaza    connected


In [6]:
# Mozambique flood mapping with GEE
import ee
import geemap

ee.Initialize(project='tamil-nadu-flood-risk')

# Mozambique bounding box
mozambique = ee.Geometry.Rectangle([32.0, -26.5, 40.5, -10.5])

# Load Sentinel-1 SAR
# Cyclone Idai hit March 2019 — use that period
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(mozambique) \
    .filterDate('2019-03-01', '2019-04-30') \
    .select('VV')

print('Sentinel-1 images found:', s1.size().getInfo())

# Median composite
s1_median = s1.median().clip(mozambique)

# Flood mask
flood_mask = s1_median.lt(-15)

# JRC historical water
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(mozambique)

# GFD flood events
gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
flood_freq = gfd.select('flooded').sum().clip(mozambique)

print('✅ All datasets loaded!')

# Display map
Map = geemap.Map()
Map.centerObject(mozambique, zoom=5)

Map.addLayer(
    s1_median,
    {'min': -25, 'max': 0,
     'palette': ['#021C3B', 'white', '#2d8a4e']},
    'Sentinel-1 VV'
)
Map.addLayer(
    flood_mask.updateMask(flood_mask),
    {'palette': ['#ff0000']},
    'Flood pixels (Cyclone Idai 2019)'
)
Map.addLayer(
    jrc,
    {'min': 0, 'max': 100,
     'palette': ['white', 'lightblue', '#0d47a1']},
    'JRC Water Occurrence'
)

display(Map)

Sentinel-1 images found: 301
✅ All datasets loaded!


Map(center=[-18.420712357968824, 36.24999999999999], controls=(WidgetControl(options=['position', 'transparent…

In [7]:
#Extract flood risk for Mozambique schools
import ee
import pandas as pd
import numpy as np

ee.Initialize(project='tamil-nadu-flood-risk')

mozambique = ee.Geometry.Rectangle([32.0, -26.5, 40.5, -10.5])

# Rebuild datasets
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(mozambique) \
    .filterDate('2019-03-01', '2019-04-30') \
    .select('VV')

s1_median = s1.median().clip(mozambique)
flood_mask = s1_median.lt(-15).rename('flood_pixel')

jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(mozambique)

gfd = ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
flood_freq = gfd.select('flooded').sum().clip(mozambique)

# Build buffered school features
features = []
for _, row in moz_df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    buffer = point.buffer(1000)
    features.append(ee.Feature(buffer, {
        'school_name': row['school_name'],
        'connectivity': row['connectivity'],
        'province':     row['province']
    }))

schools_ee = ee.FeatureCollection(features)

# Extract SAR flood exposure
sar_stats = flood_mask.reduceRegions(
    collection=schools_ee,
    reducer=ee.Reducer.mean(),
    scale=100
)
sar_data = [f['properties'] for f in sar_stats.getInfo()['features']]
sar_df = pd.DataFrame(sar_data)
sar_df = sar_df.rename(columns={'mean': 'sar_flood_pct'})
sar_df['sar_flood_pct'] = (
    sar_df['sar_flood_pct'].fillna(0) * 100
).round(2)

# Extract JRC occurrence
jrc_stats = jrc.reduceRegions(
    collection=schools_ee,
    reducer=ee.Reducer.mean(),
    scale=30
)
jrc_data = [f['properties'] for f in jrc_stats.getInfo()['features']]
jrc_df = pd.DataFrame(jrc_data)
jrc_df = jrc_df.rename(columns={'mean': 'water_occurrence_pct'})
jrc_df['water_occurrence_pct'] = jrc_df['water_occurrence_pct'].fillna(0).round(2)

# Extract GFD flood frequency
gfd_stats = flood_freq.reduceRegions(
    collection=schools_ee,
    reducer=ee.Reducer.mean(),
    scale=250
)
gfd_data = [f['properties'] for f in gfd_stats.getInfo()['features']]
gfd_df = pd.DataFrame(gfd_data)
gfd_df = gfd_df.rename(columns={'mean': 'flood_frequency'})
gfd_df['flood_frequency'] = gfd_df['flood_frequency'].fillna(0).round(2)

# Merge all
moz_risk = sar_df[['school_name', 'connectivity',
                    'province', 'sar_flood_pct']] \
    .merge(jrc_df[['school_name', 'water_occurrence_pct']], on='school_name') \
    .merge(gfd_df[['school_name', 'flood_frequency']], on='school_name')

# Combined flood score
moz_risk['sar_norm'] = (
    moz_risk['sar_flood_pct'] /
    max(moz_risk['sar_flood_pct'].max(), 1) * 100
).round(1)
moz_risk['gfd_norm'] = (
    moz_risk['flood_frequency'] /
    max(moz_risk['flood_frequency'].max(), 1) * 100
).round(1)

moz_risk['flood_score'] = (
    0.50 * moz_risk['water_occurrence_pct'] +
    0.30 * moz_risk['sar_norm'] +
    0.20 * moz_risk['gfd_norm']
).round(1)

# Connectivity penalty
moz_risk['conn_penalty'] = moz_risk['connectivity'].apply(
    lambda x: 15 if x == 'none' else 0
)
moz_risk['final_score'] = (
    moz_risk['flood_score'] + moz_risk['conn_penalty']
).round(1)

# Risk tier
def flood_risk(score):
    if score >= 50:   return 'CRITICAL'
    elif score >= 25: return 'HIGH'
    elif score >= 10: return 'MEDIUM'
    else:             return 'LOW'

moz_risk['risk_tier'] = moz_risk['final_score'].apply(flood_risk)
moz_risk = moz_risk.sort_values(
    'final_score', ascending=False
).reset_index(drop=True)
moz_risk['rank'] = moz_risk.index + 1

print('=== MOZAMBIQUE SCHOOL FLOOD RISK ===\n')
for _, row in moz_risk.iterrows():
    conn = '⚠' if row['connectivity'] == 'none' else ' '
    print(f"#{int(row['rank']):<2} {conn} "
          f"{row['school_name'][:35]:<35} | "
          f"Score: {row['final_score']:5.1f} | "
          f"Risk: {row['risk_tier']}")

=== MOZAMBIQUE SCHOOL FLOOD RISK ===

#1  ⚠ School Inhambane Coast              | Score: 108.2 | Risk: CRITICAL
#2  ⚠ Primary School Quelimane            | Score:  71.4 | Risk: CRITICAL
#3  ⚠ Primary School Tete                 | Score:  69.9 | Risk: CRITICAL
#4    Primary School Gaza                 | Score:  54.2 | Risk: CRITICAL
#5  ⚠ Secondary School Beira Coast        | Score:  53.9 | Risk: CRITICAL
#6  ⚠ Primary School Sofala               | Score:  49.9 | Risk: HIGH
#7  ⚠ Secondary School Zambezia           | Score:  15.5 | Risk: MEDIUM
#8  ⚠ School Cabo Delgado                 | Score:  15.0 | Risk: MEDIUM
#9    Primary School Maputo Central       | Score:   0.5 | Risk: LOW
#10   Secondary School Nampula            | Score:   0.0 | Risk: LOW


In [8]:
#  Cyclone hazard for Mozambique
import pandas as pd
import numpy as np
import requests

print('Loading IBTrACS South Indian Ocean cyclone data...')

# Download South Indian Ocean basin
url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.SI.list.v04r00.csv'

r = requests.get(url, timeout=60)
if r.status_code == 200:
    with open('ibtracs_SI.csv', 'wb') as f:
        f.write(r.content)
    print('✅ South Indian Ocean cyclone data downloaded!')
else:
    print(f'❌ Failed: {r.status_code}')

Loading IBTrACS South Indian Ocean cyclone data...
✅ South Indian Ocean cyclone data downloaded!


In [9]:
# Cyclone hazard for Mozambique schools
import pandas as pd
import numpy as np

# Load IBTrACS South Indian Ocean
si_raw = pd.read_csv('ibtracs_SI.csv', skiprows=[1], low_memory=False)

# Filter to Mozambique area
si_moz = si_raw[
    (pd.to_numeric(si_raw['LAT'], errors='coerce').between(-27, -10)) &
    (pd.to_numeric(si_raw['LON'], errors='coerce').between(30, 42))
].copy()

si_moz['LAT'] = pd.to_numeric(si_moz['LAT'], errors='coerce')
si_moz['LON'] = pd.to_numeric(si_moz['LON'], errors='coerce')
si_moz['SEASON'] = pd.to_numeric(si_moz['SEASON'], errors='coerce')

# Recent storms only
si_recent = si_moz[si_moz['SEASON'] >= 2000].copy()
print(f'✅ Mozambique cyclone records (2000+): {len(si_recent)}')
print(f'Unique storms: {si_recent.SID.nunique()}')

# Haversine function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Calculate cyclone proximity per school
hosp_lats = si_recent['LAT'].values
hosp_lons = si_recent['LON'].values

results = []
for _, row in moz_df.iterrows():
    dists = haversine(
        row['latitude'], row['longitude'],
        hosp_lats, hosp_lons
    )
    storms_200km = si_recent[dists <= 200]['SID'].nunique()
    storms_100km = si_recent[dists <= 100]['SID'].nunique()

    results.append({
        'school_name':   row['school_name'],
        'storms_200km':  storms_200km,
        'storms_100km':  storms_100km,
    })

cyc_df = pd.DataFrame(results)

# Normalise cyclone score
max_storms = cyc_df['storms_200km'].max()
cyc_df['cyclone_norm'] = (
    cyc_df['storms_200km'] / max_storms * 100
).round(1)

print('\n=== MOZAMBIQUE CYCLONE HAZARD ===\n')
for _, row in cyc_df.sort_values(
    'storms_200km', ascending=False
).iterrows():
    print(f"{row['school_name'][:35]:<35} | "
          f"200km: {row['storms_200km']:>3} | "
          f"Score: {row['cyclone_norm']:>5.1f}")

✅ Mozambique cyclone records (2000+): 1702
Unique storms: 58

=== MOZAMBIQUE CYCLONE HAZARD ===

Secondary School Beira Coast        | 200km:  15 | Score: 100.0
Primary School Quelimane            | 200km:  14 | Score:  93.3
Primary School Sofala               | 200km:  14 | Score:  93.3
Secondary School Nampula            | 200km:  13 | Score:  86.7
Secondary School Zambezia           | 200km:  12 | Score:  80.0
School Inhambane Coast              | 200km:  11 | Score:  73.3
Primary School Gaza                 | 200km:   8 | Score:  53.3
Primary School Tete                 | 200km:   5 | Score:  33.3
School Cabo Delgado                 | 200km:   4 | Score:  26.7
Primary School Maputo Central       | 200km:   3 | Score:  20.0


In [10]:
#Full Mozambique risk + Tamil Nadu comparison
import pandas as pd
import numpy as np

# Merge flood + cyclone
moz_full = moz_risk.merge(
    cyc_df[['school_name', 'cyclone_norm']],
    on='school_name'
)

# Multi-hazard score
moz_full['multi_hazard'] = (
    0.60 * moz_full['flood_score'] +
    0.40 * moz_full['cyclone_norm']
).round(1)

# Connectivity penalty
moz_full['final_multi'] = (
    moz_full['multi_hazard'] +
    moz_full['conn_penalty']
).round(1)

# Risk tier
def multi_risk(score):
    if score >= 50:   return 'CRITICAL'
    elif score >= 30: return 'HIGH'
    elif score >= 15: return 'MEDIUM'
    else:             return 'LOW'

moz_full['multi_risk'] = moz_full['final_multi'].apply(multi_risk)
moz_full = moz_full.sort_values(
    'final_multi', ascending=False
).reset_index(drop=True)
moz_full['rank'] = moz_full.index + 1

print('=== MOZAMBIQUE MULTI-HAZARD RISK ===')
print('Flood(60%) + Cyclone(40%) + Connectivity penalty\n')
print(f"{'#':<3} {'School':<35} {'Conn':<5} "
      f"{'Flood':>6} {'Cyc':>6} {'Score':>7} {'Risk'}")
print('-' * 72)
for _, row in moz_full.iterrows():
    conn = 'No' if row['connectivity'] == 'none' else 'Yes'
    print(
        f"#{int(row['rank']):<2} "
        f"{row['school_name'][:34]:<35} "
        f"{conn:<5} "
        f"{row['flood_score']:>5.1f} "
        f"{row['cyclone_norm']:>6.1f} "
        f"{row['final_multi']:>6.1f} "
        f"{row['multi_risk']}"
    )

print()
print('=== MOZAMBIQUE vs TAMIL NADU COMPARISON ===')
print(f"{'Metric':<40} {'Mozambique':>12} {'Tamil Nadu':>12}")
print('-' * 65)
metrics = [
    ('Schools analysed',
     len(moz_full), 15),
    ('CRITICAL risk schools',
     len(moz_full[moz_full.multi_risk == 'CRITICAL']), 7),
    ('Schools with no connectivity',
     len(moz_full[moz_full.connectivity == 'none']), 7),
    ('Average flood score',
     round(moz_full.flood_score.mean(), 1),
     round(39.8, 1)),
    ('Average cyclone score',
     round(moz_full.cyclone_norm.mean(), 1),
     round(62.5, 1)),
]
for label, moz_val, tn_val in metrics:
    print(f"{label:<40} {str(moz_val):>12} {str(tn_val):>12}")

=== MOZAMBIQUE MULTI-HAZARD RISK ===
Flood(60%) + Cyclone(40%) + Connectivity penalty

#   School                              Conn   Flood    Cyc   Score Risk
------------------------------------------------------------------------
#1  School Inhambane Coast              No     93.2   73.3  100.2 CRITICAL
#2  Primary School Quelimane            No     56.4   93.3   86.2 CRITICAL
#3  Secondary School Beira Coast        No     38.9  100.0   78.3 CRITICAL
#4  Primary School Sofala               No     34.9   93.3   73.3 CRITICAL
#5  Primary School Tete                 No     54.9   33.3   61.3 CRITICAL
#6  Primary School Gaza                 Yes    54.2   53.3   53.8 CRITICAL
#7  Secondary School Zambezia           No      0.5   80.0   47.3 HIGH
#8  Secondary School Nampula            Yes     0.0   86.7   34.7 HIGH
#9  School Cabo Delgado                 No      0.0   26.7   25.7 MEDIUM
#10 Primary School Maputo Central       Yes     0.5   20.0    8.3 LOW

=== MOZAMBIQUE vs TAMIL NADU CO

In [11]:
# Cell 8 — Fixed — rebuild moz_full then build map
import pandas as pd
import numpy as np
import folium

# Rebuild moz_full from scratch
moz_risk_data = {
    'school_name': [
        'School Inhambane Coast',
        'Primary School Quelimane',
        'Primary School Tete',
        'Primary School Gaza',
        'Secondary School Beira Coast',
        'Primary School Sofala',
        'Secondary School Zambezia',
        'School Cabo Delgado',
        'Primary School Maputo Central',
        'Secondary School Nampula'
    ],
    'latitude': [
        -23.86, -17.88, -16.16, -24.05, -19.84,
        -19.52, -17.05, -12.37, -25.96, -15.12
    ],
    'longitude': [
        35.38, 36.89, 33.59, 34.40, 34.84,
        34.56, 36.98, 40.52, 32.57, 39.27
    ],
    'connectivity': [
        'none', 'none', 'none', 'connected', 'none',
        'none', 'none', 'none', 'connected', 'connected'
    ],
    'flood_score': [
        93.2, 56.4, 54.9, 54.2, 38.9,
        34.9, 0.5, 0.0, 0.5, 0.0
    ],
    'cyclone_norm': [
        73.3, 93.3, 33.3, 53.3, 100.0,
        93.3, 80.0, 26.7, 20.0, 86.7
    ],
    'final_multi': [
        100.2, 86.2, 61.3, 53.8, 78.3,
        73.3, 47.3, 25.7, 8.3, 34.7
    ],
    'multi_risk': [
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'HIGH'
    ]
}

moz_full = pd.DataFrame(moz_risk_data)

# Tamil Nadu data
tn_data = {
    'school_name': [
        'School Puducherry Border',
        'Panchayat School Nagapattinam',
        'School Ramanathapuram',
        'School Kanchipuram',
        'Panchayat School Tirunelveli',
        'Govt School Cuddalore',
        'School Villupuram',
        'School Tuticorin',
        'Govt High School Chennai',
        'Govt School Tiruchirappalli',
        'Govt School Thanjavur',
        'High School Vellore',
        'Govt School Salem',
        'Govt School Madurai',
        'High School Coimbatore'
    ],
    'latitude': [
        11.93, 10.76, 9.37, 12.83, 8.71,
        11.75, 11.93, 8.80, 13.08, 10.79,
        10.78, 12.92, 11.65, 9.93, 11.01
    ],
    'longitude': [
        79.83, 79.84, 78.83, 79.70, 77.69,
        79.75, 79.49, 78.15, 80.27, 78.68,
        79.13, 79.13, 78.16, 78.12, 76.96
    ],
    'risk': [
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
        'HIGH', 'HIGH', 'HIGH', 'HIGH', 'MEDIUM'
    ],
    'score': [
        68.7, 68.3, 67.0, 66.6, 65.9,
        65.6, 62.3, 54.6, 44.8, 41.1,
        33.7, 29.5, 24.7, 23.3, 18.7
    ],
    'country': ['Tamil Nadu'] * 15
}

tn_df = pd.DataFrame(tn_data)

# Mozambique map data
moz_map_df = moz_full[[
    'school_name', 'latitude', 'longitude',
    'multi_risk', 'final_multi'
]].copy()
moz_map_df.columns = [
    'school_name', 'latitude', 'longitude',
    'risk', 'score'
]
moz_map_df['country'] = 'Mozambique'

# Combine both
combined = pd.concat([tn_df, moz_map_df], ignore_index=True)

# Build map
m = folium.Map(
    location=[0, 60],
    zoom_start=3,
    tiles='CartoDB positron'
)

def get_color(risk):
    return {
        'CRITICAL': '#cc0000',
        'HIGH':     '#ff6600',
        'MEDIUM':   '#ffaa00',
        'LOW':      '#2d8a4e'
    }.get(risk, 'gray')

def get_radius(score):
    if score >= 50:   return 18
    elif score >= 30: return 13
    elif score >= 15: return 9
    else:             return 6

# Add schools
for _, row in combined.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=get_radius(row['score']),
        color=get_color(row['risk']),
        fill=True,
        fill_color=get_color(row['risk']),
        fill_opacity=0.85,
        tooltip=f"{row['country']} - {row['school_name']}",
        popup=folium.Popup(
            f"<b>{row['school_name']}</b><br>"
            f"<hr style='margin:4px 0'>"
            f"🌍 {row['country']}<br>"
            f"📊 Score: {row['score']}<br>"
            f"🎯 Risk: <b>{row['risk']}</b>",
            max_width=250
        )
    ).add_to(m)

# Country labels
folium.Marker(
    [10.5, 78.5],
    icon=folium.DivIcon(
        html='<div style="font-size:14px;font-weight:bold;'
             'color:#065A82">Tamil Nadu</div>'
    )
).add_to(m)

folium.Marker(
    [-18.0, 35.0],
    icon=folium.DivIcon(
        html='<div style="font-size:14px;font-weight:bold;'
             'color:#065A82">Mozambique</div>'
    )
).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;
     background:white;padding:14px;border-radius:10px;
     border:1px solid #ccc;font-size:12px;z-index:1000;">
     <b>Multi-Country School Risk</b><br>
     <b>Tamil Nadu + Mozambique</b><br><br>
     <span style='color:#cc0000;font-size:16px'>●</span> Critical<br>
     <span style='color:#ff6600;font-size:16px'>●</span> High<br>
     <span style='color:#ffaa00;font-size:16px'>●</span> Medium<br>
     <span style='color:#2d8a4e;font-size:16px'>●</span> Low<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('mozambique_tamil_nadu_combined_map.html')
print('✅ Combined map saved!')
display(m)

✅ Combined map saved!


In [12]:
# Cell 9 — Save all Day 12 outputs
from google.colab import files

# Save CSVs
moz_full.to_csv('mozambique_risk_scores.csv', index=False)
combined.to_csv('combined_tamil_nadu_mozambique.csv', index=False)

# Download
files.download('mozambique_tamil_nadu_combined_map.html')
files.download('mozambique_risk_scores.csv')
files.download('combined_tamil_nadu_mozambique.csv')

print('✅ All files downloaded!')
print()
print('=== DAY 12 SUMMARY ===')
print(f"Mozambique schools analysed: {len(moz_full)}")
print(f"CRITICAL risk: {len(moz_full[moz_full.multi_risk == 'CRITICAL'])}")
print(f"HIGH risk: {len(moz_full[moz_full.multi_risk == 'HIGH'])}")
print(f"No connectivity: {len(moz_full[moz_full.connectivity == 'none'])}")
print()
print('Combined pipeline covers:')
print('  🇮🇳 Tamil Nadu — 15 schools')
print('  🇲🇿 Mozambique — 10 schools')
print('  🌍 Total — 25 schools across 2 countries')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!

=== DAY 12 SUMMARY ===
Mozambique schools analysed: 10
CRITICAL risk: 6
HIGH risk: 2
No connectivity: 7

Combined pipeline covers:
  🇮🇳 Tamil Nadu — 15 schools
  🇲🇿 Mozambique — 10 schools
  🌍 Total — 25 schools across 2 countries
